In [21]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [22]:
DB_USER = "postgres"
DB_PASSWORD = "your_password"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [23]:
query = """
SELECT
    order_id,
    product_id
FROM order_items
"""

order_items = pd.read_sql(
    query,
    engine
)

order_items.head()

,order_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089


In [24]:
order_items.shape

(112650, 2)

In [25]:
product_counts = order_items["product_id"].value_counts()

frequent_products = product_counts[product_counts >= 50].index

filtered_orders = order_items[
    order_items["product_id"].isin(frequent_products)
]

basket = pd.crosstab(
    filtered_orders["order_id"],
    filtered_orders["product_id"]
)
basket.head()

product_id,0152f69b6cf919bcdaf117aa8c43e5a2,054515fd15bc1a2029f10de97ffa9120,06c6e01186af8b98ee1fc9e01f9471e9,06edb72f1e0c64b14c5b79353f7abea3,08574b074924071f4e201e151b152b4e,0a57f7d2c983bcf8188589a5fea4a8da,0aabfb375647d9738ad0f7b4ea3653b1,0bcc3eeca39e1064258aa1e932269894,0d85c435fd60b277ffb9e9b0f88f927a,11875b30b49585209e608f40e8082e65,...,f35927953ed82e19d06ad3aac2f06353,f40876e0ef3cd5f9132b1f16b04b1346,f4f67ccaece962d013a4e1d7dc3a61f7,f71973c922ccaab05514a36a8bc741b8,f71f42e2381752836563b70beb542f80,f7a17d2c51d9df89a4f1711c4ac17f33,fb55982be901439613a95940feefd9ee,fbc1488c1a1e72ba175f53ab29a248e8,fbce4c4cb307679d89a3bf3d3bb353b9,fc1d8637c0268af3db482c14b7ef8e75
order_id,,,,,,,,,,,,,,,,,,,,,
00061f2a7bc09da83e415a52dc8a4af1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0006ec9db01a64e59a68b2c340bf65a7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0008288aa423d2a3f00fcb17cd7d8719,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
00130c0eee84a3d909e75bc08c5c3ca1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
001862358bf858722e1e2ae000cfed8b,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
# converting into binary values
basket = (basket > 0).astype(bool)

basket.head()

product_id,0152f69b6cf919bcdaf117aa8c43e5a2,054515fd15bc1a2029f10de97ffa9120,06c6e01186af8b98ee1fc9e01f9471e9,06edb72f1e0c64b14c5b79353f7abea3,08574b074924071f4e201e151b152b4e,0a57f7d2c983bcf8188589a5fea4a8da,0aabfb375647d9738ad0f7b4ea3653b1,0bcc3eeca39e1064258aa1e932269894,0d85c435fd60b277ffb9e9b0f88f927a,11875b30b49585209e608f40e8082e65,...,f35927953ed82e19d06ad3aac2f06353,f40876e0ef3cd5f9132b1f16b04b1346,f4f67ccaece962d013a4e1d7dc3a61f7,f71973c922ccaab05514a36a8bc741b8,f71f42e2381752836563b70beb542f80,f7a17d2c51d9df89a4f1711c4ac17f33,fb55982be901439613a95940feefd9ee,fbc1488c1a1e72ba175f53ab29a248e8,fbce4c4cb307679d89a3bf3d3bb353b9,fc1d8637c0268af3db482c14b7ef8e75
order_id,,,,,,,,,,,,,,,,,,,,,
00061f2a7bc09da83e415a52dc8a4af1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
0006ec9db01a64e59a68b2c340bf65a7,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
0008288aa423d2a3f00fcb17cd7d8719,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
00130c0eee84a3d909e75bc08c5c3ca1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
001862358bf858722e1e2ae000cfed8b,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [27]:
# running apriori algorithm to find frequent itemsets
frequent_items = apriori(
    basket,
    min_support=0.001,
    use_colnames=True
)

frequent_items.head()

,support,itemsets
0,0.003503,frozenset({0152f69b6cf919bcdaf117aa8c43e5a2})
1,0.002718,frozenset({054515fd15bc1a2029f10de97ffa9120})
2,0.003442,frozenset({06c6e01186af8b98ee1fc9e01f9471e9})
3,0.007851,frozenset({06edb72f1e0c64b14c5b79353f7abea3})
4,0.005858,frozenset({08574b074924071f4e201e151b152b4e})


In [28]:
rules = association_rules(
    frequent_items,
    metric="lift",
    min_threshold=1
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({35afc973633aaeb6b877ff57b2793310}),frozenset({99a4788cb24856965c36a24e339b6058}),0.009421,0.028204,0.001751,0.185897,6.591199,1.0,0.001486,1.193702,0.856351,0.048822,0.162270,0.123998
1,frozenset({99a4788cb24856965c36a24e339b6058}),frozenset({35afc973633aaeb6b877ff57b2793310}),0.028204,0.009421,0.001751,0.062099,6.591199,1.0,0.001486,1.056165,0.872902,0.048822,0.053178,0.123998
2,frozenset({e53e557d5a159f5aa2c5e995dfdf244b}),frozenset({36f60d45225e60c7da4558b070ce4b60}),0.009421,0.006704,0.002053,0.217949,32.511666,1.0,0.001990,1.270117,0.978460,0.145923,0.212671,0.262128
3,frozenset({36f60d45225e60c7da4558b070ce4b60}),frozenset({e53e557d5a159f5aa2c5e995dfdf244b}),0.006704,0.009421,0.002053,0.306306,32.511666,1.0,0.001990,1.427977,0.975783,0.145923,0.299709,0.262128
4,frozenset({4fcb3d9a5f4871e8362dfedbdb02b064}),frozenset({f4f67ccaece962d013a4e1d7dc3a61f7}),0.005375,0.003382,0.001027,0.191011,56.477929,1.0,0.001009,1.231931,0.987602,0.132812,0.188266,0.247291


In [29]:
recommendations = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

In [30]:
recommendations["antecedents"] = (
    recommendations["antecedents"]
    .astype(str)
)

recommendations["consequents"] = (
    recommendations["consequents"]
    .astype(str)
)

In [31]:
recommendations.to_sql(
    "product_recommendations",
    con=engine,
    if_exists="replace",
    index=False
)

6

In [32]:
recommendations = recommendations.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

In [33]:
recommendations.to_csv(
    "../trained_models/product_recommendation/association_rules.csv",
    index=False
)